# Simple classifier example

This notebook shows how to:
1. Set up a synthetic label image and feature table to try out the plugin
2. Apply a previously trained classifier to new data in a script (batch mode)

Open napari with this notebook to annotate objects and train a classifier interactively,
then use the scripted section at the bottom to apply it to new data.

## 1. Create sample data and open napari

Run the cells below to create a synthetic label image and feature table,
then open the napari-feature-classifier plugin.

In [1]:
import numpy as np
import pandas as pd
import napari

In [ ]:
# Create a synthetic 2D label image with 16 objects (each 10x10 pixels)
shape = (100, 100)
lbl_img = np.zeros(shape, dtype='uint16')
positions = [
    (slice(10, 20), slice(10, 20)),  # 1
    (slice(30, 40), slice(10, 20)),  # 2
    (slice(50, 60), slice(10, 20)),  # 3
    (slice(10, 20), slice(30, 40)),  # 4
    (slice(30, 40), slice(30, 40)),  # 5
    (slice(50, 60), slice(30, 40)),  # 6
    (slice(70, 80), slice(30, 40)),  # 7
    (slice(70, 80), slice(50, 60)),  # 8
    (slice(10, 20), slice(70, 80)),  # 9
    (slice(50, 60), slice(50, 60)),  # 10
    (slice(50, 60), slice(70, 80)),  # 11
    (slice(10, 20), slice(50, 60)),  # 12
    (slice(30, 40), slice(50, 60)),  # 13
    (slice(30, 40), slice(70, 80)),  # 14
    (slice(70, 80), slice(10, 20)),  # 15
    (slice(70, 80), slice(70, 80)),  # 16
]
for label, (row, col) in enumerate(positions, start=1):
    lbl_img[row, col] = label

In [5]:
# Feature table: two clearly separable features for easy annotation
# feature1 increases with label, feature2 decreases — two natural classes
roi_id = 'my_image'
labels = list(range(1, 17))
features = pd.DataFrame({
    'roi_id':   [roi_id] * 16,
    'label':    labels,
    'index':    labels,  # Needed for correct hover overlays in napari
    'feature1': [100, 200, 300, 500, 900, 1001, 1100, 1200, 1300, 1400, 1500, 1700, 1900, 2100, 2500, 3000],
    'feature2': [2200, 2100, 2000, 1500, 1300, 1001, 1100, 1200, 1300, 1400, 1500, 900, 800, 700, 600, 500],
})
features

,roi_id,label,index,feature1,feature2
0,my_image,1,1,100,2200
1,my_image,2,2,200,2100
2,my_image,3,3,300,2000
3,my_image,4,4,500,1500
4,my_image,5,5,900,1300
5,my_image,6,6,1001,1001
6,my_image,7,7,1100,1100
7,my_image,8,8,1200,1200
8,my_image,9,9,1300,1300
9,my_image,10,10,1400,1400


In [6]:
# Open napari and attach features to the label layer
viewer = napari.Viewer()
label_layer = viewer.add_labels(lbl_img, name='my_image')
label_layer.features = features

INFO: Adding features for roi='my_image'...
Adding features for roi='my_image'...
INFO: Training classifier...
Training classifier...
INFO: F1 score on test set: 0.0 
Annotations split into 6 training and 1 test samples. 
Training set contains {'Class_1': np.int64(3), 'Class_2': np.int64(3)}. 
Test set contains {'Class_2': np.int64(1)}.
F1 score on test set: 0.0 
Annotations split into 6 training and 1 test samples. 
Training set contains {'Class_1': np.int64(3), 'Class_2': np.int64(3)}. 
Test set contains {'Class_2': np.int64(1)}.
INFO: Saving classifier at my_image_classifier.clf...
Saving classifier at my_image_classifier.clf...
INFO: Classifier saved at my_image_classifier.clf
Classifier saved at my_image_classifier.clf


### In napari:
1. Go to `Plugins → napari-feature-classifier → Initialize a Classifier`
2. Select `feature1` and `feature2`, give your classes names (e.g. "low", "high"), click **Initialize**
3. Click objects to annotate them, then click **Run Classifier**
4. Use **▶ Saving & Export → Save Classifier As…** to save the `.clf` file
5. Use **▶ Saving & Export → Export Results As…** to export predictions to CSV

## 2. Apply a trained classifier to new data (batch / scripted mode)

Once you have trained and saved a classifier with the napari plugin,
you can apply it to new feature tables without opening napari.

The feature table must have `roi_id` and `label` columns plus all features
the classifier was trained on.

In [ ]:
import pickle
from pathlib import Path

# Path to the saved classifier (adjust to your actual path)
classifier_path = Path('my_image.clf')

# Load the classifier
with open(classifier_path, 'rb') as f:
    clf = pickle.loads(f.read())

print('Classifier trained on features:', clf.get_feature_names())
print('Classes:', clf.get_class_names())

In [ ]:
# Build a new feature table for a different image
# Must include roi_id and label, plus all features used during training
new_features = pd.DataFrame({
    'roi_id':   ['new_image'] * 16,
    'label':    list(range(1, 17)),
    'feature1': np.random.randint(100, 3000, size=16),
    'feature2': np.random.randint(100, 3000, size=16),
})

# predict() returns a Series indexed by (roi_id, label)
predictions = clf.predict(new_features)
new_features['prediction'] = predictions.values
new_features